In [ ]:
import pandas as pd 
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyClassifier #prueba de cordura de inteligencia de modelo. 


#leemos data frame original. 
from pathlib import Path
base_candidates = [Path('Machine Learning') / 'data', Path('data')]
for base in base_candidates:
    if (base / 'users_behavior.csv').exists():
        df = pd.read_csv(base / 'users_behavior.csv')
        break
else:
    raise FileNotFoundError('No se encontró users_behavior.csv en Machine Learning/data o data/')


In [2]:
#separamos en 3 df, entrenamiento, test y validacion 
df_train, df_valid=train_test_split(df, test_size=.4, random_state=54321)

df_valid, df_test=train_test_split(df_valid, test_size=.5, random_state=54321)
#para entreanamiento 
features_train= df_train.drop(['is_ultra'], axis=1)
target_train= df_train['is_ultra']

#para validacion 
features_valid= df_valid.drop(['is_ultra'], axis=1)
target_valid=df_valid['is_ultra']

#para test
features_test= df_test.drop(['is_ultra'], axis=1)
target_test=df_test['is_ultra']


In [3]:
#pruebas con modelo DesicionTreeClassifier
best_score = -1.0
best_depth = None
for depth in range(1, 11):
    model = DecisionTreeClassifier(random_state=54321, max_depth=depth)
    model.fit(features_train, target_train)
    train_acc = model.score(features_train, target_train)
    valid_acc = model.score(features_valid, target_valid)
if valid_acc > best_score:
    best_score = valid_acc
    best_depth = depth
 
print(f"Mejor profundidad según validación: {best_depth} (valid_acc={best_score:.3f})")

Mejor profundidad según validación: 10 (valid_acc=0.782)


In [4]:

#pruebas con modelo RandomForestClassifier
best_score = 0
best_est = 0
for est in range(1, 11): # selecciona el rango del hiperparámetro
    model_forest = RandomForestClassifier(random_state=54321, n_estimators=est) # configura el número de árboles
    model_forest.fit(features_train,target_train) # entrena el modelo en el conjunto de entrenamiento
    score = model.score(features_valid,target_valid) # calcula la puntuación de accuracy en el conjunto de validación
    if score > best_score:
        best_score = score# guarda la mejor puntuación de accuracy en el conjunto de validación
        best_est = est# guarda el número de estimadores que corresponden a la mejor puntuación de exactitud

print("La exactitud del mejor modelo en el conjunto de validación (n_estimators = {}): {}".format(best_est, best_score))


La exactitud del mejor modelo en el conjunto de validación (n_estimators = 1): 0.7822706065318819


In [5]:
#pruebas con modelo LogisticRegression
model_regression=LogisticRegression(random_state=54321, solver='liblinear')
model.fit(features_train,target_train)
score_train = model.score(features_train, target_train) # calcula la puntuación de accuracy en el conjunto de entrenamiento
score_valid = model.score(features_valid, target_valid) # calcula la puntuación de accuracy en el conjunto de validación

print("Accuracy del modelo de regresión logística en el conjunto de entrenamiento:", score_train)
print("Accuracy del modelo de regresión logística en el conjunto de validación:", score_valid)

Accuracy del modelo de regresión logística en el conjunto de entrenamiento: 0.883298755186722
Accuracy del modelo de regresión logística en el conjunto de validación: 0.7822706065318819


Como podemos obersvar en todas las pruebas obteners un accuracy score similar lo que puede significar que los datos no requieren un modelo de aprendizaje muy profundo o avanzado, sobre todo para optimizar el funcionamiento.

A continuacion se hacen las pruebas de calidad para comprobar el accuracy score en los datos de prueba.


In [6]:
#unimos los df de train y valid para entrenar el modelo con los parametros encontrados en las peurbas 
df_full_tr= pd.concat([df_train,df_valid])

#definimos features y target
features_full = df_full_tr.drop(['is_ultra'], axis=1)
target_full = df_full_tr['is_ultra']

In [7]:
#prueba de calidad con los paramtros obtenidos en los test: DesicionTreeClassifier
final_modelTree= DecisionTreeClassifier(random_state=54321, max_depth=10)
final_modelTree.fit(features_full, target_full)
final_test_acc= final_modelTree.score(features_test,target_test)
print("exactitud final con hiperparametros obtenidos en pruebas: DecisionTree:", final_test_acc)


exactitud final con hiperparametros obtenidos en pruebas: DecisionTree: 0.80248833592535


In [8]:
#prueba de calidad con los paramtros obtenidos en los test: RandonmForestClassifier
final_modelForest=RandomForestClassifier(random_state=54321, n_estimators=1)
final_modelForest.fit(features_full, target_full)
final_forest_acc= final_modelForest.score(features_test, target_test)
print("exactitud final con hiperparametros obtenidos en pruebas: RandomForest: ", final_test_acc)


exactitud final con hiperparametros obtenidos en pruebas: RandomForest:  0.80248833592535


In [10]:
#prueba de calidad con los paramtros obtenidos en los test: LogisticRegression
final_modelLinearReg= LogisticRegression(random_state=54321, solver='liblinear')
final_modelLinearReg.fit(features_full, target_full)
final_logis_acc= final_modelLinearReg.score(features_test, target_test)
print("exactitud final con hiperparametros obtenidos en pruebas: LogisticRegression: ", final_test_acc)


exactitud final con hiperparametros obtenidos en pruebas: LogisticRegression:  0.80248833592535


Como podemos obersvar en todos los modelos superamos el umbral de 75% de acuraccy score, sin embargo notamos nuevamente el patron de que se obtinen los mismos valores para los diferentes modelos, lo que podria significar que el modelo no esta aprendinedo si no solo replicando la distribucion de clases. 

A continuacion  se realiza una prueba de cordura para comprobar que el modelo realmente representa aprendizaje y aporta valor aprendiendo los patrones y relaciones para poder predecir de manera adecuada. 

In [13]:
dummy = DummyClassifier(strategy="most_frequent")  # siempre predice la clase mayoritaria
dummy.fit(features_train, target_train)
dummy_preds = dummy.predict(features_test)
dummy_acc = accuracy_score(target_test, dummy_preds)

print("Accuracy Dummy:", dummy_acc)
print("Accuracy Mejor Modelo:", final_test_acc)

Accuracy Dummy: 0.7247278382581649
Accuracy Mejor Modelo: 0.80248833592535


De acuerdo a los resultados de la prueba dummy, la cual siempre predice lo mismo(clase mayoritatira), el modelo si super el umbrar de 72% contra el 80% de los demas modelos. Lo cual nos indica que el modelo si esta aprendiendo patrones de las caractesiticas dadas. 


A continuacion se agregan nuevas pruebas con los modelos RandomForest y LinearRegression para ver si pueden mejorar cambiando los parametros y si vale la pena en el caso de randomForest por el tema de potencia de computo. 


In [15]:
#pruebas con modelo RandomForestClassifier: extra
best_score = 0
best_est = 0
for est in range(1, 20): # selecciona el rango del hiperparámetro
    model_forest = RandomForestClassifier(random_state=54321, n_estimators=est) # configura el número de árboles
    model_forest.fit(features_train,target_train) # entrena el modelo en el conjunto de entrenamiento
    score = model.score(features_valid,target_valid) # calcula la puntuación de accuracy en el conjunto de validación
    if score > best_score:
        best_score = score# guarda la mejor puntuación de accuracy en el conjunto de validación
        best_est = est# guarda el número de estimadores que corresponden a la mejor puntuación de exactitud

print("La exactitud del mejor modelo en el conjunto de validación (n_estimators = {}): {}".format(best_est, best_score))

La exactitud del mejor modelo en el conjunto de validación (n_estimators = 1): 0.7822706065318819


In [16]:
#pruebas con modelo LogisticRegression
model_regression=LogisticRegression(random_state=54321, solver='lgfgs')
model.fit(features_train,target_train)
score_train = model.score(features_train, target_train) # calcula la puntuación de accuracy en el conjunto de entrenamiento
score_valid = model.score(features_valid, target_valid) # calcula la puntuación de accuracy en el conjunto de validación

print("Accuracy del modelo de regresión logística en el conjunto de entrenamiento:", score_train)
print("Accuracy del modelo de regresión logística en el conjunto de validación:", score_valid)

Accuracy del modelo de regresión logística en el conjunto de entrenamiento: 0.883298755186722
Accuracy del modelo de regresión logística en el conjunto de validación: 0.7822706065318819


Conclusiones finales: Después de diversas pruebas observamos que el dataset es lineal o muy simple, por lo que la solución más adecuada puede ser Logistic Regression o Decision Tree, ya que no requieren gran poder de cómputo y entregan resultados equivalentes a modelos más complejos como Random Forest.

En todos los casos, la exactitud en el conjunto de prueba (con división 80:20) se mantiene cercana al 88%, lo cual se considera un intervalo confiable para las predicciones del modelo.